# 09 - Chat Loop Như Một State Machine

Notebook cuối không định nghĩa lại router, VLM hay retrieval. Nó chứng minh cách các khối production phối hợp trong một lượt chat và in **đầy đủ output trung gian** để debug.


## BƯỚC 1: Setup Và Import Theo Trách Nhiệm


In [ ]:
import json
import sys
import time
import uuid
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "app" / "core" / "intent.py").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy Chatbot_Fashion")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.core.intent import get_clarify_response, get_out_of_scope_response, get_social_response, route_user_request
from app.core.profile import apply_profile_decision, sanitize_profile_candidate
from app.core.security import CommerceFactStreamFilter, validate_user_query


## BƯỚC 2: State Có Cấu Trúc, Không Giữ Ảnh Raw

`pending_image_docs/context` là dữ liệu suy ra. File ảnh/base64 không được đưa vào state. `pending_profile_candidate` cũng chưa phải profile cho tới khi user xác nhận.


In [ ]:
def create_chat_state(session_id: str | None = None) -> dict:
    '''Tạo toàn bộ state cần cho routing follow-up và profile confirmation.'''
    return {
        "session_id": session_id or str(uuid.uuid4()),
        "profile": {},
        "last_bot_msg": "",
        "last_route_decision": None,
        "last_query": "",
        "pending_profile_candidate": None,
        "pending_image_context": None,
        "pending_image_docs": [],
        "unclear_count": 0,
    }

chat_state = create_chat_state("notebook-09")
print(json.dumps(chat_state, ensure_ascii=False, indent=2, default=str))


## BƯỚC 3: Chuẩn Bị Lượt Có Ảnh Theo Hai Pha

Pha 1 dùng text/modality. Chỉ khi action là `inspect_image` mới gọi VLM observation. Pha 2 route lại với observation. Cách này tránh gọi VLM thừa khi user đã nói rõ “phối đồ” hoặc “phân tích dáng”.


In [ ]:
def prepare_image_turn(image_path: str | Path, user_input: str, state: dict, execute_models: bool = False) -> dict:
    '''Chuẩn bị decision ảnh; model nặng chỉ chạy khi `execute_models=True`.'''
    path = Path(image_path)
    decision = route_user_request(user_input, state=state, has_image=True)
    result = {"decision_before_vision": decision.to_debug_dict(), "image_path_valid": path.exists()}
    if decision.action == "inspect_image":
        if not execute_models:
            result["status"] = "needs_vision_observation"
            return result
        from app.core.vision import describe_image_for_routing
        observation = describe_image_for_routing(str(path), user_input)
        decision = route_user_request(user_input, state=state, has_image=True, image_context=observation)
        result["image_observation"] = observation
    result["decision"] = decision.to_debug_dict()
    result["status"] = "ready"
    return result


## BƯỚC 4: Control Handler Dừng Sớm Trước Khi Load Chain


In [ ]:
def resolve_control_response(decision, state: dict) -> str | None:
    '''Xử lý route không cần retrieval/LLM answer chain.'''
    if decision.handler == "profile_management":
        reply, _ = apply_profile_decision(decision, state)
        return reply
    if decision.handler == "social":
        return get_social_response(decision.action)
    if decision.handler == "out_of_scope":
        return get_out_of_scope_response(decision.rewrite_query)
    if decision.handler == "clarify":
        return get_clarify_response(decision)
    return None


## BƯỚC 5: Một Lượt Chat Hoàn Chỉnh Ở Chế Độ Debug

Mặc định hàm chỉ lập kế hoạch (`execute=False`) nên Run All không tải model. Khi bật execute, nhánh ảnh chạy vision/image retrieval; nhánh trả lời gọi đúng production chain. Kết quả luôn chứa decision đầy đủ.


In [ ]:
def run_chat_turn(user_input: str, state: dict, image_path: str | Path | None = None, execute: bool = False) -> dict:
    '''Validate → route → state/control → optional execution, với output debug đầy đủ.'''
    started = time.time()
    valid, message = validate_user_query(user_input or "")
    if not valid:
        return {"status":"rejected", "answer":message, "elapsed_sec":round(time.time()-started, 4)}

    image_docs = []
    temporary_profile = None
    if image_path:
        prepared = prepare_image_turn(image_path, user_input, state, execute_models=execute)
        if prepared["status"] != "ready":
            return {**prepared, "elapsed_sec":round(time.time()-started, 4)}
        decision = route_user_request(
            user_input, state=state, has_image=True,
            image_context=prepared.get("image_observation"),
        )
        if execute and decision.handler == "profile_analysis":
            from app.core.vision import analyze_person_image
            raw = analyze_person_image(str(image_path))
            candidate = sanitize_profile_candidate(raw)
            state["pending_profile_candidate"] = candidate
            temporary_profile = {**state["profile"], **candidate}
            if decision.action != "analyze_then_style":
                answer = f"Candidate tạm: {candidate}. Bạn có đồng ý lưu không?"
                return {"status":"awaiting_profile_confirmation", "decision":decision.to_debug_dict(), "candidate":candidate, "answer":answer}
        elif execute and not decision.needs_clarification:
            from app.core.image_search import search_products_by_image
            image_docs = search_products_by_image(str(image_path))
            state["pending_image_docs"] = image_docs
            state["pending_image_context"] = decision.image_context
    else:
        decision = route_user_request(user_input, state=state, last_bot_msg=state.get("last_bot_msg", ""))
        if decision.route == "image_outfit_advice":
            image_docs = state.get("pending_image_docs", [])

    control_answer = resolve_control_response(decision, state)
    result = {"status":"planned", "decision":decision.to_debug_dict(), "answer":control_answer, "executed":False}
    if control_answer is not None or not execute:
        if control_answer:
            state["last_bot_msg"] = control_answer
        if decision.handler in {"search", "image_search", "outfit"}:
            state["last_route_decision"] = decision
            state["last_query"] = decision.rewrite_query or user_input
        result["elapsed_sec"] = round(time.time()-started, 4)
        return result

    active_query = decision.rewrite_query or user_input
    if decision.handler in {"search", "image_search"}:
        if decision.handler == "image_search":
            from app.core.chains import get_product_answer_chain
            from app.core.llm import format_documents_for_llm
            response = get_product_answer_chain().invoke({"input":active_query, "context":format_documents_for_llm(image_docs)})
        else:
            from app.core.chains import get_fast_search_chain
            response = get_fast_search_chain().invoke({"input":active_query})
        answer = response.get("answer", response) if isinstance(response, dict) else getattr(response, "content", str(response))
    else:
        from app.core.chains import get_outfit_chain
        from app.core.outfit import build_outfit_context, build_outfit_context_from_image_docs
        profile = temporary_profile or state["profile"]
        gender = profile.get("gender", "female")
        if image_docs:
            context, images, diagnostics = build_outfit_context_from_image_docs(image_docs, active_query, gender, profile)
        else:
            context, images = build_outfit_context(active_query, gender, profile)
            diagnostics = None
        response = get_outfit_chain().invoke({"input":active_query, "outfit_context":context})
        answer = getattr(response, "content", str(response))
        result.update({"images":images, "outfit_diagnostics":diagnostics})

    # Product card là nguồn sự thật; bỏ các dòng mã/giá/brand/ảnh do LLM tự viết.
    fact_filter = CommerceFactStreamFilter()
    answer = fact_filter.feed(answer + "\n") + fact_filter.finish()
    result["filtered_commerce_fact_lines"] = fact_filter.removed_lines
    if decision.follow_up_question:
        answer += "\n\n" + decision.follow_up_question
    if decision.action == "analyze_then_style":
        answer += "\n\nProfile vẫn là candidate tạm. Bạn có đồng ý lưu không?"
    state["last_route_decision"] = decision
    state["last_query"] = active_query
    state["last_bot_msg"] = answer[-1200:]
    result.update({"status":"completed", "answer":answer, "executed":True, "elapsed_sec":round(time.time()-started, 4)})
    return result


## BƯỚC 6: Smoke Test Không Gọi Dịch Vụ Ngoài


In [ ]:
smoke_state = create_chat_state("smoke")
for query in ["Xin chào", "tìm áo sơ mi trắng", "phối đồ đi làm", "xem thêm"]:
    result = run_chat_turn(query, smoke_state, execute=False)
    print("\n===", query, "===")
    print(json.dumps(result, ensure_ascii=False, indent=2, default=str))

assert run_chat_turn("Xin chào", create_chat_state(), execute=False)["decision"]["intent"] == "social"
print("[PASS] Chat-loop smoke test")


## BƯỚC 7: Terminal UI Là Lớp Mỏng

`/debug on` bật execution thật; `/image <path> | <câu hỏi>` truyền ảnh local; `/profile` xem state. Logic AI vẫn nằm trong các hàm production.


In [ ]:
def start_chat_loop(state: dict | None = None) -> dict:
    state = state or create_chat_state()
    execute = False
    print("Lệnh: /debug on|off, /profile, /image <path> | <câu hỏi>, exit")
    while True:
        raw = input("Bạn: ").strip()
        if raw.lower() in {"exit", "quit"}:
            return state
        if raw.startswith("/debug "):
            execute = raw.split(maxsplit=1)[1].lower() == "on"
            print("execute=", execute)
            continue
        if raw == "/profile":
            print(json.dumps(state["profile"], ensure_ascii=False, indent=2))
            continue
        image_path = None
        message = raw
        if raw.startswith("/image "):
            payload = raw[len("/image "):]
            image_path, _, message = payload.partition("|")
            image_path, message = image_path.strip(), message.strip()
        result = run_chat_turn(message, state, image_path=image_path, execute=execute)
        print(json.dumps(result, ensure_ascii=False, indent=2, default=str))


## Kết Luận: Đọc Theo Trạng Thái Chuyển Tiếp

Một lượt đi qua `rejected → needs_vision → clarify/control → planned → completed`. Khi debug, đừng hỏi chung “chatbot sai ở đâu”; hãy xem nó dừng ở trạng thái nào, decision nào, context nào. Đây là lợi ích lớn nhất của việc tách intent, modality, action và route.
